# Quickstart: nuclear outlines in 2D

Outlines every nucleus in a z-stack of an envelope staining (an ER marker, a lamin, ...), where a section through a
nucleus is a bright ring around a dark inside. `OUT_DIR` receives

- `<name>_labels.tif`: label image (0 = background, nuclei numbered largest first)
- `<name>_nuclei.csv`: one row per nucleus (area, centroid, shape, z window)
- `<name>_parameters.json`: parameters, voxel size and versions, to repeat the run
- `<name>_overview.png`, `<name>_gallery.png`: outlines on the image, to check the result

Set `STACK_PATH` to your TIFF z-stack; without one, a small synthetic stack is used. Expected input: see
the README. How it works and how to tune it: `2_method_2d.ipynb`. Volumes: `3_quickstart_3d.ipynb`.

In [ ]:
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

import nucseg

In [ ]:
STACK_PATH = None            # your TIFF z-stack, e.g. "data/stack.tif"; None = synthetic example
OUT_DIR = Path("results")
CHANNEL = None               # channel of multichannel files (0-based); None for single-channel files
UM_PER_PIXEL = None          # pixel size in µm; None reads it from the ImageJ/OME metadata

path = STACK_PATH or nucseg.write_synthetic_stack(Path("example_data") / "synthetic.tif")
stack = nucseg.read_stack(path, channel=CHANNEL, um_per_pixel=UM_PER_PIXEL)
z_step = "unknown" if stack.z_step_um is None else f"{stack.z_step_um:.3f} µm"
print(f"{stack.name}: {stack.data.shape[0]} slices of {stack.data.shape[1]} x {stack.data.shape[2]} px, "
      f"{stack.um_per_pixel:.4f} µm/px, z step {z_step}")

## Parameters

The pixel size is read from the file (set `UM_PER_PIXEL` if it is missing or wrong) and intensities are scaled
automatically. Set the size limits to your nuclei; everything else has defaults (`help(nucseg.Parameters)`).

In [ ]:
parameters = nucseg.Parameters(
    bin_factor=4,          # pixels averaged per axis, for speed; too coarse loses thin rings
    window=3,              # slices averaged per projection, about 1 µm
    min_area_um2=25.0,     # size of your nuclei
    max_area_um2=800.0,
    min_width_um=5.0,
    z_first=None,          # first and last slice used (inclusive); None = first / last
    z_last=None,
    border_frame=True,     # also nuclei cut by the image edge, marked border=True (default False)
)

## Segmentation

About 35-40 s per 2048 x 2048 x 20 stack (Apple M4 Pro). Check each overview and gallery; if nuclei are missing, split
or merged, see the table *Tuning* at the end of `2_method_2d.ipynb`.

In [ ]:
segmentation = nucseg.segment_2d(stack, parameters)
nucseg.save_results(OUT_DIR, stack, segmentation)
print(f"{len(segmentation.nuclei)} nuclei")
display(Image(OUT_DIR / f"{stack.name}_overview.png"), Image(OUT_DIR / f"{stack.name}_gallery.png"))

## Nuclei

One row per nucleus; `label` is its value in the label image. Leave `border = True` nuclei out of size statistics:
their area is only the visible part.

In [ ]:
nuclei = pd.DataFrame(segmentation.table()).set_index("label")
nuclei

In [ ]:
nuclei[~nuclei.border]["area_um2"].describe().round(1)